In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
results_silver_path = (
    "s3://bizf1project123/Silver/Aaisha/incremental-load/results"
)

sprints_silver_path = (
    "s3://bizf1project123/Silver/Aaisha/incremental-load/sprints"
)

target_path = (
    "s3://bizf1project123/Gold/Aaisha/incremental-load/fact_session_results"
)

In [0]:
results_df = (
    spark.read.format("delta")
    .load(results_silver_path)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("RACE"))
    .withColumnRenamed("finish_position", "final_position")
    .withColumnRenamed("finish_position_text", "final_position_text")
    .drop(
        "race_name",
        "race_date",
        "ingestion_timestamp",
        "source_file",
        "created_timestamp",
        "updated_timestamp"
    )
)

In [0]:
sprints_df = (
    spark.read
    .format("delta")
    .load(sprints_silver_path)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("SPRINT"))
    .drop(
        "race_name",
        "race_date",
        "ingestion_timestamp",
        "source_file",
        "created_timestamp",
        "updated_timestamp"
    )
)

display(sprints_df)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

display(results_sprints_df)

In [0]:
fact_session_results_df = (
    results_sprints_df
    .withColumn("is_win", F.col("final_position") == 1)
    .withColumn("is_podium", F.col("final_position").between(1, 3))
    .withColumn("has_points", F.col("points") > 0)
    .withColumn("created_timestamp", F.current_timestamp())
    .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
display(
    fact_session_results_df
    .filter(F.col("season") == 2025)
    .orderBy("round", "session_type", "final_position")
)

In [0]:
if not DeltaTable.isDeltaTable(spark, target_path):
    (
        fact_session_results_df.write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )
else:
    target_table = DeltaTable.forPath(spark, target_path)

    (
        target_table.alias("t")
        .merge(
            fact_session_results_df.alias("s"),
            """
            t.season = s.season
            AND t.round = s.round
            AND t.constructor_id = s.constructor_id
            AND t.driver_id = s.driver_id
            AND t.session_type = s.session_type
            """
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "grid_position": "s.grid_position",
                "completed_laps": "s.completed_laps",
                "car_number": "s.car_number",
                "points": "s.points",
                "final_position": "s.final_position",
                "final_position_text": "s.final_position_text",
                "status": "s.status",
                "is_win": "s.is_win",
                "is_podium": "s.is_podium",
                "has_points": "s.has_points",
                "batch_id": "s.batch_id",
                "updated_timestamp": "s.updated_timestamp"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
fact_session_results = spark.read.format("delta").load(target_path)

print("Total rows:", fact_session_results.count())

display(
    fact_session_results
    .orderBy("season", "round", "session_type", "final_position")
)